# 阶段 7：手动切换三种增强模式

写 `MODE_SEL`（0x04）即可切换。硬件会在**下一个帧边界**才让新模式生效，
所以随便什么时候写都不会撕裂画面。

本 notebook 会把三种模式各抓一帧放在一起对比。

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib.ai_video_regs import AiVideoRegs

# 三档模式对比必须在**采集时那一档锁定曝光**下做，否则输入画面本身就变了，
# 对比会变成「曝光差异」而不是「处理差异」（采集工作点 = 736 行 + 增益 0x20）。
# 画面太暗/全黑时先改下面这个开关：False = 退回自动曝光（画面会亮起来，
# 但特征会偏移、三类中心对不上）。
USE_MANUAL_EXPOSURE = True
DEMO_EXPOSURE, DEMO_GAIN = 736, 0x20      # 采集时锁定的工作点，重采过就改这里
ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(
    ROOT, manual_exposure=(dict(exposure=DEMO_EXPOSURE, gain=DEMO_GAIN)
                           if USE_MANUAL_EXPOSURE else None))
vs.stop_all_channels(vdma0, vdma1)
regs = AiVideoRegs(ol.axis_video_proc_0)
print("VERSION = 0x%08X  (阶段 %d)" % (regs.version(), regs.version_stage()))
assert regs.version_stage() >= 7, "当前 overlay 不是阶段 7 的硬件"

vp.start_capture(vdma0, mode)
regs.wait_new_frame()

## 三种模式各抓一帧

注意：抓到的画面是**处理之后**的（IP 插在 demosaic 与 VDMA0 之间），
所以它能直接反映模式的视觉效果——这也正好用来做验证。

In [ ]:
MODE_NAMES = ["0 dark-boost x1.5", "1 backlight-compress", "2 detail-sharpen"]
shots = {}

for m in (0, 1, 2):
    regs.set_mode(m)
    # 等至少 3 帧，保证 mode_active 已经在帧边界更新
    for _ in range(3):
        regs.wait_new_frame()
    shots[m] = vp.frame_to_rgb(vp.capture_frame(vdma0))
    print("mode %d (%s) 已抓帧，均值亮度 = %.1f"
          % (m, MODE_NAMES[m], shots[m].mean()))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, (0, 1, 2)):
    ax.imshow(shots[m])
    ax.set_title("MODE_SEL=%d  %s" % (m, MODE_NAMES[m]))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 看一眼“平均值”是否和预期一致

mode 0（×1.5）的平均亮度应该明显高于 mode 2（锐化不改平均亮度）。

In [ ]:
means = {m: float(shots[m].mean()) for m in (0, 1, 2)}
print("平均像素值:", {m: round(v, 1) for m, v in means.items()})
assert means[0] > means[2], "mode0 应该比 mode2 亮；不成立说明像素通路没接对"
print("模式切换验证通过")

## 复验：差异到底是“模式造成的”还是“场景自己变了”

上面三张图是**先后三次**抓的，中间隔着不同时间。如果镜头前的东西动了、
或者摄像头的自动曝光（AEC）还在收敛，均值变化里就混进了“场景自己变了”这一项，
不能干净地证明是模式在起作用。

所以换个做法：**让场景不动**，按 `0,1,2,0,1,2` 的顺序各抓一帧，
看同一个模式两次读到的是不是同一个值。

* 两次接近 → 画面差异确实是模式造成的（模式真的生效了）；
* 两次差很多 → 是场景/曝光在漂，换个不动的场景重跑这一格。

顺带确认阶段 6 的功能没被破坏（`MLP_OUTPUT` 仍可读、帧率仍然正常）。

In [ ]:
# 复验：模式真的生效了吗？
# 场景不动，按 0,1,2,0,1,2 各抓一帧。同一个模式两次读到的值应当接近。
ORDER = (0, 1, 2, 0, 1, 2)
seq = []
for m in ORDER:
    regs.set_mode(m)
    for _ in range(3):          # 等 3 帧，确保 mode_active 已在帧边界更新
        regs.wait_new_frame()
    seq.append((m, float(vp.frame_to_rgb(vp.capture_frame(vdma0)).mean())))

per_mode = {}
for m, v in seq:
    per_mode.setdefault(m, []).append(v)

print('抓帧顺序:', ['%d:%.1f' % (m, v) for m, v in seq])
for m in (0, 1, 2):
    a, b = per_mode[m]
    print('mode %d 两次均值 %.1f / %.1f，相差 %.1f' % (m, a, b, abs(a - b)))

# 阶段 6 的功能还在不在：MLP 出结果 + 硬件帧率正常
import time
c0, t0 = regs.frame_cnt(), time.time()
time.sleep(1.0)
fps = (regs.frame_cnt() - c0) / (time.time() - t0)

m0 = sum(per_mode[0]) / len(per_mode[0])
m1 = sum(per_mode[1]) / len(per_mode[1])
m2 = sum(per_mode[2]) / len(per_mode[2])
print('平均亮度: mode0 %.1f, mode1 %.1f, mode2 %.1f' % (m0, m1, m2))
print('MLP_OUTPUT = %d, 帧率 ≈ %.1f fps' % (regs.mlp_output(), fps))

# 1) 硬检查：mode0 是逐点 ×1.5（饱和），mode1 的曲线逐点都不超过它，
#    所以 m0 >= m1 在设计上必然成立；不成立只可能是像素处理没接进视频通路。
assert m0 >= m1 - 1e-6, 'mode0 应该不低于 mode1；不成立说明像素处理没接进通路'
# 2) 硬检查：流水线还活着（FRAME_CNT 由 PL 的像素时钟驱动，正常约 60/秒）
assert fps > 5, '帧率只有 %.1f fps，视频通路可能已经停了' % fps

# 3) 软提示：同一模式两次的差异，就是「场景 / 自动曝光漂移」的量级
drift = max(abs(per_mode[m][0] - per_mode[m][1]) for m in (0, 1, 2))
if drift <= 6.0:
    print('✅ 可重复性 OK：同一模式两次最多差 %.1f 灰度，模式效果稳定' % drift)
else:
    print('⚠️ 同一模式两次差了 %.1f 灰度，偏大：这几百毫秒里场景 / 自动曝光变了，'
          '请对着不动的场景再跑一次这个格子。' % drift)
if m0 - m1 < 2.0:
    print('提示：mode0 和 mode1 几乎一样，是因为画面偏暗（每个通道都 < 96）——'
          '这两种模式在暗场景下按设计就是同一条曲线，属于正常；'
          '想看出区别请换到正常光照下再跑一次。')
# 阈值定 26 而不是 30：720p30 标称就是 30 fps，实测 27~30 都算正常。
# 原来写成 < 30，把正常的 29.9 也报成「偏低」，答辩时屏幕上多一行要解释的噪声。
if fps < 26:
    print('提示：帧率偏低（%.1f fps），确认没有别的程序在抢 CPU / 带宽。' % fps)

## 诊断：差别出在“处理”还是“输入画面”

`LUMA_SUM / PIXEL_CNT` 是 IP 在**处理前**的输入像素上统计的（不是抓到的画面）。
用它就能把两件事分开：

* 三档的“输入亮度”基本一样、而 mode2 的“输出均值”明显低 → 是**处理逻辑**把画面压暗了；
* mode2 的“输入亮度”本身就低 → 是**摄像头曝光/增益（AEC）**变了，处理逻辑没问题。

“水平差分”= 相邻像素差值绝对值的平均，越大说明高频（锐化、也包括噪声）越强。

In [ ]:
# 诊断：三档各自的「输入」和「输出」
# LUMA_SUM / PIXEL_CNT 是 IP 在**处理前**的输入像素上统计的，
# 用它就能分辨「处理把画面压暗了」还是「摄像头曝光被改了」。
def diag(mode, warm=5):
    regs.set_mode(mode)
    for _ in range(warm):
        regs.wait_new_frame()
    s = regs.raw_stats()
    raw_luma = s['luma_sum'] / max(s['pixel_cnt'], 1)
    f = vp.frame_to_rgb(vp.capture_frame(vdma0)).astype(np.float32)
    return (mode, raw_luma, float(f.mean()), float(f.std()),
            float(np.abs(np.diff(f, axis=1)).mean()),
            100.0 * float((f == 0).mean()),
            100.0 * float((f >= 255).mean()))

print('模式 | 输入亮度(IP统计) | 输出均值 | 输出std | 水平差分 | 0占比 | 满255占比')
for r in [diag(m) for m in (0, 1, 2)]:
    print('  %d  | %14.1f | %8.1f | %7.1f | %8.1f | %5.1f%% | %6.1f%%' % r)

## 收尾

演示前建议把模式恢复到 0 或你想要的默认值：

```python
regs.set_mode(0)
```

下一阶段（阶段 8）会让 Jupyter 根据 `MLP_OUTPUT` **自动**切换，不需要手动写寄存器。

In [ ]:
import gc

def safe_shutdown(ol=None, vdma0=None, vdma1=None, vs=None):
    """收尾：停 VDMA 通道 + 尽量释放 overlay。任何一步失败都不影响后面。

    为什么不再写 demosaic.stop / iic.stop：
    v_demosaic_0 和 axi_iic_0 只是 AXI-Lite 寄存器空间，**根本没有 .stop() 方法**。
    上一版写成 _try(demosaic.stop, ...) 会报 AttributeError —— 因为属性取值
    （demosaic.stop）发生在调用 _try 之前，异常在“传参那一刻”就抛出来了，
    _try 里面的 try 根本来不及接住。真正需要停的只有 VDMA 通道。
    """
    def _try(fn, name):
        try:
            fn()
            print('[ok]   %s' % name)
        except Exception as e:
            print('[warn] %s: %s' % (name, e))

    # 1) 停 VDMA 通道（唯一必须做的一步）
    if vs is not None and vdma0 is not None:
        _try(lambda: vs.stop_all_channels(vdma0, vdma1), 'stop_all_channels')
    elif vdma0 is not None:
        _try(vdma0.readchannel.stop, 'vdma0.readchannel.stop')

    # 2) 释放 overlay（PYNQ 各版本接口名不一样，失败也只是打个 warn）
    if ol is not None:
        _try(lambda: ol.free(), 'overlay.free')

    gc.collect()
    print('✅ 硬件资源已释放')

safe_shutdown(ol=ol, vdma0=vdma0, vdma1=vdma1, vs=vs)